In [19]:
import pandas as pd
import json
from dotenv import load_dotenv
import os
from supabase import create_client
from datetime import datetime, timedelta

load_dotenv()

supbase_url = os.getenv("SUPABASE_URL")
supbase_key = os.getenv("SUPABASE_KEY")

supabase = create_client(supbase_url, supbase_key)


# Filings

In [ ]:
df_filings = data = (
    supabase
    .table("idx_filings")
    .select("*").gte("timestamp", "2026-05-01")
    .execute()
)

df_filings = pd.DataFrame(df_filings.data)

df_filings = df_filings[["source","created_at",'symbol',"title",'tags','price','transaction_value','holding_before','holding_after','share_percentage_before','share_percentage_after','share_percentage_transaction','holder_name','context', 'price_transaction']]

notes: for the content we will have these kind of schema

1. daily filings (generated daily, one content per day, with all filings of that day)
2. context/mesop (generated when it triggered, one company per image)

## Daily

In [29]:
# Convert created_at to datetime if not already
df_filings['created_at'] = pd.to_datetime(df_filings['created_at'])

# Get records from last 24 hours (using pandas Timestamp with UTC timezone)
last_24h = pd.Timestamp.now(tz='UTC') - timedelta(hours=120)
df_filings_daily = df_filings[df_filings['created_at'] >= last_24h]

In [30]:
df_filings_daily = df_filings_daily[(df_filings_daily["transaction_value"]>100000000) & (df_filings_daily["share_percentage_transaction"] >= 0.5)]

In [32]:
df_filings_daily.to_csv("filings_daily.csv", index=False)

In [31]:
df_filings_daily

,source,created_at,symbol,title,tags,price,transaction_value,holding_before,holding_after,share_percentage_before,share_percentage_after,share_percentage_transaction,holder_name,context,price_transaction
55,https://www.idx.co.id/StaticData/NewsAndAnnoun...,2026-05-13 06:44:33.180957+00:00,PACK.JK,H Samsudin Andi Arsyad buys shares of Abadi Nu...,[investment],137.000,936650874900,0,6836867700,0.000,21.120,21.120,H Samsudin Andi Arsyad,3 insiders bought PACK in the last 6 months.,"[{'date': '2026-05-13', 'type': 'buy', 'price'..."
56,https://www.idx.co.id/StaticData/NewsAndAnnoun...,2026-05-13 06:44:33.180957+00:00,NSSS.JK,Kurniadi Patriawan buys shares of Nusantara Sa...,[investment],800.000,380800000000,13098500,489098500,0.060,2.060,2.000,Kurniadi Patriawan,NaN,"[{'date': '2026-05-08', 'type': 'buy', 'price'..."
57,https://www.idx.co.id/StaticData/NewsAndAnnoun...,2026-05-13 06:44:33.180957+00:00,NSSS.JK,Samuel Tumbuh Bersama sells shares of Nusantar...,[divestment],800.000,1048000000000,8342811900,7032811900,35.050,29.550,5.500,Samuel Tumbuh Bersama,NaN,"[{'date': '2026-05-08', 'type': 'sell', 'price..."
60,https://www.idx.co.id/StaticData/NewsAndAnnoun...,2026-05-13 10:18:15.731709+00:00,AMRT.JK,Amanda Cipta Persada buys shares of Sumber Alf...,[capital-restructuring],1415.000,2996615684000,3710794900,5828544500,8.940,14.040,5.100,Amanda Cipta Persada,NaN,"[{'date': '2026-05-13', 'type': 'buy', 'price'..."
78,https://www.idx.co.id/StaticData/NewsAndAnnoun...,2026-05-13 17:57:56.720591+00:00,CUAN.JK,Prajogo Pangestu sells shares of Petrindo Jaya...,[free_float_compliance],1035.000,1035000000000,91353198700,90353198700,81.261,80.372,0.889,Prajogo Pangestu,8th insider sell by Prajogo Pangestu in the la...,"[{'date': '2026-05-11', 'type': 'sell', 'price..."
79,https://www.idx.co.id/StaticData/NewsAndAnnoun...,2026-05-13 10:18:15.731709+00:00,AMRT.JK,Sigmantara Alfindo sells shares of Sumber Alfa...,[capital-restructuring],1415.000,2996615684000,18269532859,16151783259,44.000,38.900,5.100,Sigmantara Alfindo,NaN,"[{'date': '2026-05-13', 'type': 'sell', 'price..."
80,https://www.idx.co.id/StaticData/NewsAndAnnoun...,2026-05-15 02:34:28.011146+00:00,SRSN.JK,Ludijanto Setijo sells shares of Indo Acidatama,[divestment],67.106,3303997400,49235312,12,0.817,0.000,0.817,Ludijanto Setijo,4th insider sell by Ludijanto Setijo in the la...,"[{'date': '2026-05-12', 'type': 'sell', 'price..."


## Context

please add for top 100 companies by mcap only

In [33]:
df_filings_context = df_filings[~df_filings.context.isnull()]
df_filings_context

,source,created_at,symbol,title,tags,price,transaction_value,holding_before,holding_after,share_percentage_before,share_percentage_after,share_percentage_transaction,holder_name,context,price_transaction
1,https://www.idx.co.id/StaticData/NewsAndAnnoun...,2026-05-09 06:11:58.451090+00:00,HEAL.JK,Yulisar Khiat buys shares of Medikaloka Hermina,[investment],1008.000,1512000000,986829166,988329166,6.420,6.430,0.010,Yulisar Khiat,3rd insider buy by Yulisar Khiat in the last 6...,"[{'date': '2026-05-08', 'type': 'buy', 'price'..."
2,https://www.idx.co.id/StaticData/NewsAndAnnoun...,2026-05-08 11:07:07.472434+00:00,MEDS.JK,Jemmy Kurniawan sells shares of Hetzer Medical...,[divestment],88.063,132094200,912570900,911070900,58.400,58.310,0.090,Jemmy Kurniawan,22nd insider sell by Jemmy Kurniawan in the la...,"[{'date': '2026-05-06', 'type': 'sell', 'price..."
3,https://www.idx.co.id/StaticData/NewsAndAnnoun...,2026-05-08 11:07:07.472434+00:00,WINR.JK,Pemenang Nusantara Internasional sells shares ...,[divestment],30.060,381488800,3198050400,3185359400,61.090,60.840,0.250,Pemenang Nusantara Internasional,14th insider sell by Pemenang Nusantara Intern...,"[{'date': '2026-05-05', 'type': 'sell', 'price..."
6,https://www.idx.co.id/StaticData/NewsAndAnnoun...,2026-05-09 06:11:58.451090+00:00,HEAL.JK,Hasmoro buys shares of Medikaloka Hermina,[investment],1023.000,1299005400,756718089,757987889,4.920,4.930,0.010,Hasmoro,7th insider buy by Hasmoro in the last 6 months.,"[{'date': '2026-05-06', 'type': 'buy', 'price'..."
7,https://www.idx.co.id/StaticData/NewsAndAnnoun...,2026-05-09 06:11:58.451090+00:00,HEAL.JK,Hasmoro buys shares of Medikaloka Hermina,[investment],1000.000,1488000000,757987889,759475889,4.930,4.940,0.010,Hasmoro,8th insider buy by Hasmoro in the last 6 months.,"[{'date': '2026-05-08', 'type': 'buy', 'price'..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144,https://www.idx.co.id/StaticData/NewsAndAnnoun...,2026-05-06 06:26:40.248090+00:00,PNGO.JK,Charles Sutantio sells shares of Pinago Utama,[divestment],3584.000,493146572800,137596700,0,17.610,0.000,17.610,Charles Sutantio,3 insiders sold PNGO in the last 6 months.,"[{'date': '2026-05-04', 'type': 'sell', 'price..."
145,https://www.idx.co.id/StaticData/NewsAndAnnoun...,2026-05-07 10:19:20.244892+00:00,AKRA.JK,Termurti Tiban buys shares of AKR Corporindo,[investment],499.000,1197600000,1650000,4050000,0.008,0.020,0.012,Termurti Tiban,5 insiders bought AKRA in the last 6 months.,"[{'date': '2026-05-06', 'type': 'buy', 'price'..."
146,https://www.idx.co.id/StaticData/NewsAndAnnoun...,2026-05-07 10:19:20.244892+00:00,AKRA.JK,Nery Polim buys shares of AKR Corporindo,[investment],499.000,1796400000,2965000,6565000,0.015,0.033,0.018,Nery Polim,6 insiders bought AKRA in the last 6 months.,"[{'date': '2026-05-06', 'type': 'buy', 'price'..."
147,https://www.idx.co.id/StaticData/NewsAndAnnoun...,2026-05-07 10:19:20.244892+00:00,AKRA.JK,Vembu Suresh buys shares of AKR Corporindo,[investment],499.000,1197600000,2812800,5212800,0.014,0.026,0.012,Vembu Suresh,7 insiders bought AKRA in the last 6 months.,"[{'date': '2026-05-06', 'type': 'buy', 'price'..."


In [39]:
def extract_context_pattern(ctx):
    if 'bought' in ctx.lower():
        return 'Insider Buy'
    elif 'sell' in ctx.lower():
        return 'Insider Sell'
    else:
        return 'Other'

df_filings_context = df_filings_context.copy()
df_filings_context['context_pattern'] = df_filings_context['context'].apply(extract_context_pattern)

df_grouped = (
    df_filings_context
    .groupby(['symbol', 'context_pattern'])
    .agg(
        count=('context', 'count'),
        holders=('holder_name', lambda x: list(x.dropna().unique())),
        latest_context=('context', 'last'),
        latest_price=('price', 'last'),
        latest_transaction_value=('transaction_value', 'last'),
        latest_share_pct_before=('share_percentage_before', 'last'),
        latest_share_pct_after=('share_percentage_after', 'last'),
        latest_holding_before=('holding_before', 'last'),
        latest_holding_after=('holding_after', 'last'),
    )
    .reset_index()
)

df_grouped


,symbol,context_pattern,count,holders,latest_context,latest_price,latest_transaction_value,latest_share_pct_before,latest_share_pct_after,latest_holding_before,latest_holding_after
0,AKPI.JK,Insider Sell,4,[Henry Liem],27th insider sell by Henry Liem in the last 6 ...,522.235,235058000,2.227,2.153,13634259,13184159
1,AKRA.JK,Insider Buy,5,"[Bambang Soetiono S, Termurti Tiban, Nery Poli...",8 insiders bought AKRA in the last 6 months.,499.000,5988000000,0.229,0.289,46000000,58000000
2,ASII.JK,Insider Buy,1,[Prijono Sugiarto],6 insiders bought ASII in the last 6 months.,5925.000,1054650000,0.010,0.011,4123300,4301300
3,ASLI.JK,Other,1,[Wahana Konstruksi Mandiri],3 insiders sold ASLI in the last 6 months.,272.000,199240000000,62.720,51.000,3920000000,3187500000
4,ATLA.JK,Insider Sell,3,[Rudi R Sutantra],18th insider sell by Rudi R Sutantra in the la...,51.000,1020000000,44.350,44.020,2749254200,2729254200
5,BIKE.JK,Other,1,[Andrew Mulyadi],3 insiders sold BIKE in the last 6 months.,0.000,0,25.300,0.000,327375000,0
6,BMRI.JK,Insider Buy,1,[Sunarto],3 insiders bought BMRI in the last 6 months.,4445.000,3556000000,0.000,0.000,1000000,1800000
7,BULL.JK,Other,2,[Wong Kevin],14th insider buy by Wong Kevin in the last 6 m...,494.000,988000000,2.200,2.210,340837950,342837950
8,BYAN.JK,Insider Sell,3,[Oliver Khaw Kar Heng],15th insider sell by Oliver Khaw Kar Heng in t...,11145.158,1860126900,0.000,0.000,166900,0
9,BYAN.JK,Other,1,[Alastair Gordon Christopher Mcleod],4 insiders sold BYAN in the last 6 months.,12000.000,36000000000,0.021,0.012,7000000,4000000


## MESOP & Take Over

please add for top 100 companies by mcap only

In [45]:
import ast

IMPORTANT_TAGS = {
    'takeover',
    'capital-restructuring',
    'repurchase-agreement',
    'free_float_compliance',
    'mesop',
}

def parse_tags(t):
    if isinstance(t, list):
        return t
    if isinstance(t, str):
        try:
            return ast.literal_eval(t)
        except Exception:
            return [t]
    return []

def has_important_tag(tags):
    return bool(set(parse_tags(tags)) & IMPORTANT_TAGS)

df_important = df_filings[df_filings['tags'].apply(has_important_tag)].copy()
df_important['tags_parsed'] = df_important['tags'].apply(parse_tags)

df_important[['symbol', 'title', 'tags_parsed', 'holder_name', 'price', 'transaction_value',
              'holding_before', 'holding_after', 'share_percentage_before', 'share_percentage_after',
              'created_at']]


,symbol,title,tags_parsed,holder_name,price,transaction_value,holding_before,holding_after,share_percentage_before,share_percentage_after,created_at
9,CUAN.JK,Prajogo Pangestu sells shares of Petrindo Jaya...,[free_float_compliance],Prajogo Pangestu,1185.000,48687858000,91394285500,91353198700,81.298,81.261,2026-05-08 15:16:58.628383+00:00
13,MAPI.JK,Satya Mulia Gema Gemilang sells shares of Mitr...,"[divestment, takeover]",Satya Mulia Gema Gemilang,1395.000,11810070000000,8466000000,0,51.000,0.000,2026-05-11 07:38:30.678824+00:00
18,MDLA.JK,"Hetty Soetikno, Dra sells shares of PT Medela ...",[capital-restructuring],"Hetty Soetikno, Dra",220.000,1386000000000,9240000000,2940000000,65.940,20.980,2026-05-11 09:23:21.363367+00:00
28,BIKE.JK,Stephen Mulyadi sells shares of Sepeda Bersama...,[takeover],Stephen Mulyadi,0.000,0,266750000,0,20.620,0.000,2026-05-04 07:26:05.859169+00:00
30,AMMN.JK,Pesona Sukses Cemerlang sells shares of PT Amm...,[repurchase-agreement],Pesona Sukses Cemerlang,4950.000,3199999997700,4468377112,3821912466,6.162,5.270,2026-05-12 06:26:35.542741+00:00
31,BIKE.JK,Henry Mulyadi sells shares of Sepeda Bersama I...,[takeover],Henry Mulyadi,0.000,0,327375000,0,25.300,0.000,2026-05-04 07:27:54.974742+00:00
38,MAPI.JK,Pacific Universal Investments Pte Ltd buys sha...,"[share-transfer, takeover]",Pacific Universal Investments Pte Ltd,1395.000,1395000000,83660000,84660000,0.000,51.000,2026-05-12 07:06:19.098856+00:00
60,AMRT.JK,Amanda Cipta Persada buys shares of Sumber Alf...,[capital-restructuring],Amanda Cipta Persada,1415.000,2996615684000,3710794900,5828544500,8.940,14.040,2026-05-13 10:18:15.731709+00:00
62,INPS.JK,Surya Perkasa Sentosa sells shares of Indah Pr...,"[investment, takeover]",Surya Perkasa Sentosa,89.000,46156557000,518613000,0,79.790,0.000,2026-05-05 09:58:50.161945+00:00
76,INPS.JK,Graha Inti Guna Persada buys shares of Indah P...,"[investment, takeover]",Graha Inti Guna Persada,89.000,50606557000,0,568613000,0.000,87.480,2026-05-07 06:31:51.147248+00:00


# News

In [49]:
data = (
    supabase
    .table("idx_news")
    .select("*")
    .execute()
)

In [50]:
df_news = pd.DataFrame(data.data)

In [48]:
df_news

,id,created_at,title,body,source,timestamp,sector,sub_sector,tags,tickers,dimension,votes,score,symbols,thumbnail
0,14408,2026-04-15T05:49:43.147903+00:00,PT Matahari Putra Prima Tbk announces 64.9% ri...,PT Matahari Putra Prima Tbk plans a rights iss...,https://www.idnfinancials.com/news/62959/mppa-...,2026-04-14T12:06:51,consumer-non-cyclicals,[food-staples-retailing],"[Rights Issue, Capital & Funding, Business Exp...",[MPPA.JK],"{'future': 0, 'dividend': 0, 'ownership': 0, '...",None,92.0,[MPPA.JK],NaN
1,14334,2026-04-09T09:10:05.86724+00:00,Pemenang Nusantara Internasional sell Shares o...,"Pemenang Nusantara Internasional sold 39,900,0...",https://www.idx.co.id/StaticData/NewsAndAnnoun...,2026-04-09T14:25:02,properties-real-estate,[properties-real-estate],[Insider Trading],[WINR.JK],None,None,NaN,[WINR.JK],NaN
2,14371,2026-04-13T10:30:58.991941+00:00,Sandra Angela buys shares of PT Cashlez Worldw...,"Sandra Angela, an insider, bought 1,018,700 sh...",https://www.idx.co.id/StaticData/NewsAndAnnoun...,2026-04-13T09:08:00,technology,[software-it-services],[Insider Trading],[CASH.JK],None,None,NaN,[CASH.JK],NaN
3,14373,2026-04-14T05:48:43.408321+00:00,PT Matahari Putra Prima Tbk launches Rp1.19 tr...,PT Matahari Putra Prima Tbk announced a rights...,https://market.bisnis.com/read/20260414/192/19...,2026-04-14T03:32:21,consumer-non-cyclicals,"[food-staples-retailing, food-staples-retailing]","[Rights Issue, Ownership, Capital & Funding]","[MPPA.JK, MLPL.JK]","{'future': 0, 'dividend': 0, 'ownership': 2, '...",None,90.0,"[MPPA.JK, MLPL.JK]",NaN
4,14374,2026-04-14T05:48:43.408321+00:00,PT Sillo Maritime Perdana Tbk announces resign...,PT Sillo Maritime Perdana Tbk received Eddy Wi...,https://market.bisnis.com/read/20260414/192/19...,2026-04-14T03:32:21,energy,[oil-gas-coal],"[Executive Changes, Shareholders General Meeting]",[SHIP.JK],"{'future': 0, 'dividend': 0, 'ownership': 0, '...",None,85.0,[SHIP.JK],NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5223,14327,2026-04-09T06:25:03.623899+00:00,PT Yanaprima Hasta Persada Tbk hit with second...,PT Yanaprima Hasta Persada Tbk’s shares were s...,https://emitennews.com/news/bei-berlakukan-sus...,2026-04-09T00:00:00,basic-materials,[basic-materials],"[Suspension, Bullish]",[YPAS.JK],"{'future': 0, 'dividend': 0, 'ownership': 0, '...",None,65.0,[YPAS.JK],NaN
5224,14328,2026-04-09T06:25:03.623899+00:00,PT Sepeda Bersama Indonesia Tbk reports 2025 n...,PT Sepeda Bersama Indonesia Tbk announced at i...,https://emitennews.com/news/imbas-rugi-bersih-...,2026-04-09T00:00:00,consumer-cyclicals,[leisure-goods],"[Dividend Announcement, Shareholders General M...",[BIKE.JK],"{'future': 0, 'dividend': 0, 'ownership': 0, '...",None,95.0,[BIKE.JK],NaN
5225,14329,2026-04-09T06:25:03.623899+00:00,Harta Djaya suspends 2025 dividend and reports...,Harta Djaya announced at its 8 April 2026 shar...,https://emitennews.com/news/meja-rancang-sejum...,2026-04-09T00:00:00,consumer-cyclicals,[household-goods],"[Dividend Announcement, Rights Issue, Subsidia...",[MEJA.JK],"{'future': 0, 'dividend': 0, 'ownership': 0, '...",None,83.0,[MEJA.JK],NaN
5226,14330,2026-04-09T06:25:03.623899+00:00,Royaltama Mulia Kontraktorindo approves 512‑mi...,Royaltama Mulia Kontraktorindo received shareh...,https://emitennews.com/news/manajemen-baru-rmk...,2026-04-09T00:00:00,energy,[oil-gas-coal],"[Rights Issue, Executive Changes, Capital & Fu...",[RMKO.JK],"{'future': 0, 'dividend': 0, 'ownership': 0, '...",None,90.0,[RMKO.JK],NaN


In [51]:
import ast

def parse_tags(t):
    if isinstance(t, list):
        return t
    if isinstance(t, str):
        try:
            return ast.literal_eval(t)
        except Exception:
            return [t]
    return []

all_news_tags = sorted(set(
    tag
    for tags in df_news['tags'].dropna().apply(parse_tags)
    for tag in tags
))

print(f"Total unique tags: {len(all_news_tags)}")
print(all_news_tags)


Total unique tags: 77
[': Not Applicable', 'Analyst Ratings', 'Annual Report', 'Artificial Intelligence', 'Asset Management', 'Asset Purchase', 'Award', 'Bearish', 'Bonds', 'Bonus', 'Bullish', 'Business Expansion', 'Capital & Funding', 'Central Bank', 'Commodities', 'Credit', 'Cryptocurrency', 'Currency & FX', 'Cyber Security', 'Debt Issuance', 'Delisting', 'Digital Payments', 'Digital Transformation', 'Diversification', 'Dividend Announcement', 'Domestic Investor', 'ESG', 'Executive Changes', 'Export', 'Financial Metrics', 'Foreign Investment', 'Forex', 'Global Economy', 'Global Index', 'Government Policy', 'Halal', 'Housing Loan', 'IPO', 'Import', 'Inflation', 'Insider Trading', 'Institutional Investor', 'Interest Rate', 'Joint Operation', 'Joint Venture', 'Market Sentiment', 'Mergers & Acquisitions', 'Ministry', 'MoU', 'Mortgage', 'Neutral', 'OJK', 'Oversubscribed', 'Overvalued', 'Ownership', 'Partnerships & Agreements', 'Pension Fund', 'Pilot Project', 'Politics & Regulation', 'Por

In [ ]:
TIER_1_TAGS = {
    'IPO',
    'Mergers & Acquisitions',
    'Dividend Announcement',
    'Stock Buyback',
    'Stock Split',
    'Rights Issue',
    'Insider Trading',
    'Trading Halt',
    'Suspension',
    'Delisting',
}

TIER_2_TAGS = {
    'Executive Changes',
    'Business Expansion',
    'Joint Venture',
    'MoU',
    'Private Placement',
    'Oversubscribed',
    'Ownership',
    'Shareholders General Meeting',
    'Violation',
    'Foreign Investment',
    'OJK',
    'Government Policy',
}

IMPORTANT_NEWS_TAGS = TIER_1_TAGS | TIER_2_TAGS

def has_important_news_tag(tags):
    return bool(set(parse_tags(tags)) & IMPORTANT_NEWS_TAGS)

def get_tier(tags):
    parsed = set(parse_tags(tags))
    if parsed & TIER_1_TAGS:
        return 'Tier 1'
    elif parsed & TIER_2_TAGS:
        return 'Tier 2'
    return 'Other'

df_news_important = df_news[df_news['tags'].apply(has_important_news_tag)].copy()
df_news_important['tags_parsed'] = df_news_important['tags'].apply(parse_tags)
df_news_important['tier'] = df_news_important['tags'].apply(get_tier)

df_news_important = df_news_important.sort_values('tier')
df_news_important


,id,created_at,title,body,source,timestamp,sector,sub_sector,tags,tickers,dimension,votes,score,symbols,thumbnail,tags_parsed,tier
0,14408,2026-04-15T05:49:43.147903+00:00,PT Matahari Putra Prima Tbk announces 64.9% ri...,PT Matahari Putra Prima Tbk plans a rights iss...,https://www.idnfinancials.com/news/62959/mppa-...,2026-04-14T12:06:51,consumer-non-cyclicals,[food-staples-retailing],"[Rights Issue, Capital & Funding, Business Exp...",[MPPA.JK],"{'future': 0, 'dividend': 0, 'ownership': 0, '...",None,92.0,[MPPA.JK],NaN,"[Rights Issue, Capital & Funding, Business Exp...",Tier 1
3018,11713,2026-01-29T14:30:28.236215+00:00,Anita Anwar sell Shares of Sarana Menara Nusan...,"Anita Anwar sold 22,385,000 Sarana Menara Nusa...",https://www.idx.co.id/StaticData/NewsAndAnnoun...,2026-01-29T20:00:33,infrastructures,[telecommunication],[Insider Trading],[TOWR.JK],None,None,NaN,[TOWR.JK],NaN,[Insider Trading],Tier 1
3021,13833,2026-03-28T05:39:35.216634+00:00,PT Sentul City Tbk and other Indonesian firms ...,"On March 25, 2024, PT Sentul City Tbk saw its ...",https://www.idnfinancials.com/news/62518/samue...,2026-03-27T12:41:26,properties-real-estate,"[properties-real-estate, heavy-constructions-c...","[Ownership, Stock Buyback, Institutional Inves...","[BKSL.JK, SSIA.JK, BUKA.JK, WINS.JK, LEAD.JK, ...","{'future': 0, 'dividend': 0, 'ownership': 2, '...",None,90.0,"[BKSL.JK, SSIA.JK, BUKA.JK, WINS.JK, LEAD.JK, ...",NaN,"[Ownership, Stock Buyback, Institutional Inves...",Tier 1
3022,11714,2026-01-29T14:30:28.457059+00:00,Andrew Phillip Starkey Buys Shares of PT Merde...,"On 2026-01-28, investor Andrew Phillip Starkey...",https://www.idx.co.id/StaticData/NewsAndAnnoun...,2026-01-29T20:03:20,basic-materials,[basic-materials],[Insider Trading],[MDKA.JK],None,None,NaN,[MDKA.JK],NaN,[Insider Trading],Tier 1
3023,12076,2026-02-10T07:50:22.943962+00:00,Repower Asia Indonesia commits to stronger cor...,Repower Asia Indonesia said it will comply wit...,https://emitennews.com/news/perkuat-tata-kelol...,2026-02-10T00:00:00,properties-real-estate,[properties-real-estate],"[Risk & Compliance, Politics & Regulation, Ana...",[REAL.JK],"{'future': 1, 'dividend': 0, 'ownership': 0, '...",None,61.0,[REAL.JK],NaN,"[Risk & Compliance, Politics & Regulation, Ana...",Tier 1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3774,13145,2026-03-10T06:06:32.501809+00:00,Indonesian Ministry of Industry says textile a...,The Ministry of Industry reported that nationa...,https://bcasekuritas.co.id/en/latest-news/news...,2026-03-09T21:11:26,consumer-cyclicals,[apparel-luxury-goods],"[Ministry, Import, Government Policy, Politics...",[],"{'future': 1, 'dividend': 0, 'ownership': 0, '...",None,65.0,[],NaN,"[Ministry, Import, Government Policy, Politics...",Tier 2
2635,12604,2026-02-24T08:09:21.423659+00:00,PT Soho Global Health Tbk appoints former PT K...,PT Soho Global Health Tbk announced the appoin...,https://market.bisnis.com/read/20260224/192/19...,2026-02-24T05:06:29,healthcare,[pharmaceuticals-health-care-research],"[Executive Changes, Business Expansion, Divers...",[SOHO.JK],"{'future': 1, 'dividend': 0, 'ownership': 0, '...",None,70.0,[SOHO.JK],NaN,"[Executive Changes, Business Expansion, Divers...",Tier 2
2636,12605,2026-02-24T08:09:21.423659+00:00,PT Astra Otoparts Tbk posts 8.4% net profit ri...,PT Astra Otoparts Tbk reported a 8.43% YoY inc...,https://market.bisnis.com/read/20260224/192/19...,2026-02-24T04:58:29,consumer-cyclicals,"[automobiles-components, multi-sector-holdings]","[Annual Report, Financial Metrics, Business Ex...","[ASII.JK, AUTO.JK]","{'future': 2, 'dividend': 0, 'ownership': 0, '...",None,85.0,"[ASII.JK, AUTO.JK]",NaN,"[Annual Report, Financial Metrics, Business Ex...",Tier 2
2630,11621,2026-01-28T07:46:16.95462+00:00,PT Vale Indonesia Tbk advances three HPAL nick...,PT Vale Indonesia Tbk is accelerating construc...,https://indonesiaminer.com/news/detail/vale-in...,2026-01-21T08:58:00,basic-materials,"[basic-materia

notes:

1. for tier one data: generate per news, and could be triggered once when the data available in the db
2. for tier two data: could be generated daily, something like the summary of the news for that day 